# Instructor setup - Databricks side

Runs IN Databricks, on the course cluster. The companion to
`scripts/instructor_setup_aws.py`. It provisions the Databricks-side
non-terraformable artifacts that Weeks 19-23 notebooks assume already
exist: the Unity Catalog fraud data, Delta Change Data Feed, and the
shared workspace folder.

It does NOT create terraformable infrastructure. The `bread_academy`
catalog and its schemas are issued with `CREATE ... IF NOT EXISTS` so this
script is self-sufficient; if Terraform also creates them, IF NOT EXISTS
makes both safe.

Every step is GUARDED and idempotent. In particular the data load is
guarded: it loads ONLY if the table is absent or empty. A blind overwrite
would reset the Delta version and wipe the Change Data Feed history that
the Week 20 data-engineering notebook depends on.

Design: `plans/bootstrap_environment_aws_databricks.md`. Run by the
instructor once before class; students never run this.

In [ ]:
# Step 0 - configuration.
CATALOG = "bread_academy"
COURSE_SCHEMA = "course_data"
WORK_SCHEMA = "student_work"
SOURCE_TABLE = f"{CATALOG}.{COURSE_SCHEMA}.fraud_transactions"
SHARED_FOLDER = "/Shared/bread_academy"

# Synthetic data parameters - deterministic so every run is reproducible.
SEED = 42
N_DAYS = 90
ROWS_PER_DAY = 500          # ~45k rows total
FRAUD_BASE_RATE = 0.03
DRIFT_START_DAY = 60        # merchant_country distribution shifts after day 60

print("catalog:", CATALOG)
print("source table:", SOURCE_TABLE)

In [ ]:
# Step 1 - catalog and schemas.
#
# This metastore has NO default storage root, so a bare CREATE CATALOG fails
# with "Metastore storage root URL does not exist" - even CREATE CATALOG IF
# NOT EXISTS evaluates the storage location before the existence short-circuit
# and errors. So we check existence first and only CREATE (with an explicit
# MANAGED LOCATION) when the catalog is genuinely absent.
#
# MANAGED_LOCATION reuses the pre-existing 'dcacademy' external location, the
# same one the original migration used to create this catalog.
MANAGED_LOCATION = "abfss://demo@dcacademy.dfs.core.windows.net/bread_academy"

_existing = {r[0] for r in spark.sql("SHOW CATALOGS").collect()}
if CATALOG in _existing:
    print(f"catalog {CATALOG} already exists - not creating")
else:
    spark.sql(
        f"CREATE CATALOG {CATALOG} MANAGED LOCATION '{MANAGED_LOCATION}'"
    )
    print(f"created catalog {CATALOG} at {MANAGED_LOCATION}")

# Schemas: CREATE SCHEMA IF NOT EXISTS is safe once the catalog exists -
# they inherit the catalog's managed location.
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{COURSE_SCHEMA} "
    "COMMENT 'Instructor-loaded datasets - read-only for students'"
)
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{WORK_SCHEMA} "
    "COMMENT 'Student working area - read/write'"
)
print(f"schemas ready: {COURSE_SCHEMA}, {WORK_SCHEMA}")


In [ ]:
# Step 2 - load the synthetic fraud_transactions table (GUARDED).
#
# Loads ONLY if the table is absent or empty. If it already has rows we
# SKIP - re-running with a blind overwrite would reset the Delta version
# and destroy the Change Data Feed history the Week 20 DE notebook needs.

import numpy as np
import pandas as pd
from datetime import date, timedelta


def _table_has_rows():
    try:
        return spark.sql(f"SELECT COUNT(*) c FROM {SOURCE_TABLE}").collect()[0]["c"] > 0
    except Exception:
        return False


def _generate_fraud_df():
    """Deterministic 11-column synthetic fraud transactions.
    Drift injected on merchant_country after DRIFT_START_DAY."""
    rng = np.random.default_rng(SEED)
    start = date.today() - timedelta(days=N_DAYS)
    categories = ["groceries", "electronics", "fuel", "restaurants",
                  "travel", "clothing", "wire_transfer", "luxury_goods"]
    rows = []
    txn = 0
    for day in range(N_DAYS):
        pdate = start + timedelta(days=day)
        # Country mix drifts after DRIFT_START_DAY: US share drops, BR/NG rise.
        if day < DRIFT_START_DAY:
            countries, weights = ["US", "CA", "GB", "BR", "NG"], [0.80, 0.08, 0.06, 0.04, 0.02]
        else:
            countries, weights = ["US", "CA", "GB", "BR", "NG"], [0.55, 0.08, 0.07, 0.18, 0.12]
        for _ in range(ROWS_PER_DAY):
            txn += 1
            country = rng.choice(countries, p=weights)
            category = rng.choice(categories)
            hour = int(rng.integers(0, 24))
            days_since = int(rng.exponential(15))
            # Fraud over-indexes on non-US, late hours, long inactivity.
            p = FRAUD_BASE_RATE
            if country not in ("US", "CA"):
                p *= 3.0
            if hour < 6:
                p *= 2.0
            if days_since > 30:
                p *= 2.0
            is_fraud = int(rng.random() < min(p, 0.95))
            mu = np.log(450) if is_fraud else np.log(70)
            amount = round(float(rng.lognormal(mu, 1.0)), 2)
            cust = f"cust_{int(rng.integers(0, 5000)):05d}"
            narrative = (
                f"Customer {cust} ran a ${amount:.2f} {category} "
                f"transaction at {hour:02d}:00 in {country} after "
                f"{days_since} days of inactivity"
            )
            if is_fraud:
                narrative += " (flagged for review)"
            rows.append((
                f"txn_{txn:07d}", cust, amount, country, category, hour,
                int(pdate.weekday() >= 5), days_since, narrative, is_fraud,
                pdate,
            ))
    return pd.DataFrame(rows, columns=[
        "transaction_id", "customer_id", "amount", "merchant_country",
        "merchant_category", "hour_of_day", "is_weekend",
        "days_since_last_txn", "is_fraud", "narrative", "partition_date",
    ])


if _table_has_rows():
    n = spark.sql(f"SELECT COUNT(*) c FROM {SOURCE_TABLE}").collect()[0]["c"]
    print(f"SKIP: {SOURCE_TABLE} already has {n:,} rows. Not overwriting "
          "(would wipe Change Data Feed history). Drop the table manually "
          "if you intend to reload.")
else:
    pdf = _generate_fraud_df()
    # Column order in the generator tuple differs from the DataFrame columns
    # list above only cosmetically; reindex to the canonical order.
    pdf = pdf[[
        "transaction_id", "customer_id", "amount", "merchant_country",
        "merchant_category", "hour_of_day", "is_weekend",
        "days_since_last_txn", "narrative", "is_fraud", "partition_date",
    ]]
    sdf = spark.createDataFrame(pdf)
    (sdf.write.partitionBy("partition_date").mode("overwrite")
        .saveAsTable(SOURCE_TABLE))
    print(f"loaded {len(pdf):,} rows into {SOURCE_TABLE}")

In [ ]:
# Step 3 - enable Delta Change Data Feed (idempotent).
# The Week 20 data-engineering notebook reads the change feed; CDF only
# captures changes made AFTER it is enabled, so enable it now.
spark.sql(
    f"ALTER TABLE {SOURCE_TABLE} "
    "SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
print(f"Change Data Feed enabled on {SOURCE_TABLE}")

In [ ]:
# Step 4 - shared workspace folder for MLflow experiment paths (idempotent).
# The Week 20 DE notebook logs to /Shared/bread_academy/... ; the parent
# folder must exist or a nested experiment path fails.
import json
import urllib.request

_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
_token = _ctx.apiToken().get()
_req = urllib.request.Request(
    _host + "/api/2.0/workspace/mkdirs",
    data=json.dumps({"path": SHARED_FOLDER}).encode(),
    headers={"Authorization": f"Bearer {_token}",
             "Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(_req, timeout=30) as r:
    r.read()
print(f"workspace folder ready: {SHARED_FOLDER}")

In [ ]:
# Step 5 - summary.
n = spark.sql(f"SELECT COUNT(*) c FROM {SOURCE_TABLE}").collect()[0]["c"]
cdf = spark.sql(f"SHOW TBLPROPERTIES {SOURCE_TABLE}").toPandas()
cdf_on = any(
    (row["key"] == "delta.enableChangeDataFeed" and row["value"] == "true")
    for _, row in cdf.iterrows()
)
print("=" * 56)
print("INSTRUCTOR SETUP - DATABRICKS SIDE: complete")
print("=" * 56)
print(f"  catalog/schemas : {CATALOG}.{COURSE_SCHEMA}, {CATALOG}.{WORK_SCHEMA}")
print(f"  fraud table     : {SOURCE_TABLE} ({n:,} rows)")
print(f"  change data feed: {'enabled' if cdf_on else 'NOT ENABLED'}")
print(f"  workspace folder: {SHARED_FOLDER}")
print("Next: run scripts/instructor_setup_aws.py from a laptop if the "
      "AWS-side resources are not provisioned yet.")
dbutils.notebook.exit("instructor_setup_databricks OK")